# Supply Chain Deep Learning Demand Forecasting: Colab GPU Benchmark
### PyTorch Comparative Study: Multi-layer LSTM vs. BiLSTM vs. GRU
This notebook is self-contained and pre-configured to run with free Google Colab GPU (T4).

#### Pipeline Overview:
1. **Data Ingestion & Sanitization**: Currency cleaning and mixed date parsing.
2. **Feature Engineering**: Retail fulfillment lead times, revenues, and profit margins.
3. **Continuous Calendar Zero-Filling**: Expanding 47 SKUs across 945 calendar days.
4. **Sequence Generation**: Sliding window (30 lookback -> 7 horizon).
5. **Model Benchmarking**: Side-by-side training of LSTM, BiLSTM, and GRU.
6. **Enterprise Metrics**: MAE, RMSE, and WAPE (Walmart/Amazon industry standard).
7. **Checkpoint Export**: Download `demand_lstm.pt` and `scaler.pkl` directly to your local machine.


In [ ]:
# Check GPU Availability and Install Dependencies
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 1. Data Ingestion & Sanitization
Upload `US_Regional_Sales_Data.csv` or load from sample data.


In [ ]:
import os
import pandas as pd
import numpy as np

# If running on Colab, upload file if not present
if not os.path.exists('US_Regional_Sales_Data.csv'):
    from google.colab import files
    print('Please upload US_Regional_Sales_Data.csv:')
    uploaded = files.upload()

df = pd.read_csv('US_Regional_Sales_Data.csv')
print('Raw Data Shape:', df.shape)

# Sanitize dirty currency fields
for col in ['Unit Cost', 'Unit Price']:
    df[col] = df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(float)

# Robust mixed date parsing
for col in ['ProcuredDate', 'OrderDate', 'ShipDate', 'DeliveryDate']:
    df[col] = pd.to_datetime(df[col], format='mixed', dayfirst=True)

# Supply chain KPI calculations
df['total_fulfillment_days'] = (df['DeliveryDate'] - df['OrderDate']).dt.days
df['gross_revenue'] = df['Order Quantity'] * df['Unit Price']
df['discount_amount'] = df['gross_revenue'] * df['Discount Applied']
df['net_revenue'] = df['gross_revenue'] - df['discount_amount']
df['total_cost'] = df['Order Quantity'] * df['Unit Cost']
df['gross_profit'] = df['net_revenue'] - df['total_cost']

print('--- Supply Chain Baseline KPIs ---')
print(f'Net Revenue: ${df["net_revenue"].sum()/1e6:.2f} Million')
print(f'Gross Profit: ${df["gross_profit"].sum()/1e6:.2f} Million')
print(f'Operating Margin: {(df["gross_profit"].sum() / df["net_revenue"].sum()) * 100:.2f}%')
print(f'Network Avg Fulfillment: {df["total_fulfillment_days"].mean():.2f} days')


## 2. Continuous Calendar Zero-Filling Grid
Reindex daily sales to eliminate intermittent gaps and supply continuous chronological sequences.


In [ ]:
min_date = df['OrderDate'].min()
max_date = df['OrderDate'].max()
full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
skus = sorted(df['_ProductID'].unique())

daily_sales = df.groupby(['_ProductID', 'OrderDate'])['Order Quantity'].sum().reset_index()
grid_index = pd.MultiIndex.from_product([skus, full_date_range], names=['_ProductID', 'OrderDate'])
grid_df = pd.DataFrame(index=grid_index).reset_index()

merged_grid = pd.merge(grid_df, daily_sales, on=['_ProductID', 'OrderDate'], how='left')
merged_grid['Order Quantity'] = merged_grid['Order Quantity'].fillna(0.0)
print(f'Continuous grid created: {len(merged_grid)} total records ({len(skus)} SKUs x {len(full_date_range)} days).')


## 3. Sequence Generator (30 Lookback -> 7 Horizon)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader
import joblib

LOOKBACK = 30
HORIZON = 7
BATCH_SIZE = 64

scaler = MinMaxScaler(feature_range=(0, 1))
merged_grid = merged_grid.sort_values(by=['_ProductID', 'OrderDate']).copy()
merged_grid['scaled_demand'] = scaler.fit_transform(merged_grid[['Order Quantity']].values)

X_list, y_list = [], []
for sku_id, group in merged_grid.groupby('_ProductID'):
    series = group['scaled_demand'].values
    for i in range(len(series) - LOOKBACK - HORIZON + 1):
        X_list.append(series[i:i + LOOKBACK])
        y_list.append(series[i + LOOKBACK:i + LOOKBACK + HORIZON])

X_arr = np.array(X_list)[:, :, np.newaxis]
y_arr = np.array(y_list)

split_idx = int(0.80 * len(X_arr))
X_train, y_train = X_arr[:split_idx], y_arr[:split_idx]
X_test, y_test = X_arr[split_idx:], y_arr[split_idx:]

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)
print(f'Train samples: {len(X_train)}, Test samples: {len(X_test)}')


## 4. Deep Learning Architectures: LSTM vs. BiLSTM vs. GRU


In [ ]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, HORIZON))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class BiLSTMModel(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.bilstm = nn.LSTM(1, hidden_dim, num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.fc = nn.Sequential(nn.Linear(hidden_dim * 2, 32), nn.ReLU(), nn.Linear(32, HORIZON))
    def forward(self, x):
        out, _ = self.bilstm(x)
        return self.fc(out[:, -1, :])

class GRUModel(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(1, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, HORIZON))
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])


## 5. Benchmarking & Training Loop with Loss Convergence Curves


In [ ]:
import matplotlib.pyplot as plt

def train_and_eval(model, name, epochs=15, lr=0.003):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    
    print(f'Training {name} on {device}...')
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(bx)
        epoch_loss = running_loss / len(train_loader.dataset)
        loss_history.append(epoch_loss)
        if epoch % 5 == 0 or epoch == epochs:
            print(f'{name} Epoch {epoch}/{epochs} - Loss: {epoch_loss:.5f}')
            
    # Evaluation
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            bx = bx.to(device)
            p = model(bx).cpu().numpy()
            preds.append(p)
            actuals.append(by.numpy())
            
    y_pred = scaler.inverse_transform(np.vstack(preds))
    y_true = scaler.inverse_transform(np.vstack(actuals))
    y_pred = np.clip(y_pred, 0, None)
    
    # Metrics: MAE, RMSE, WAPE
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    sum_true = np.sum(np.abs(y_true))
    wape = (np.sum(np.abs(y_true - y_pred)) / sum_true) * 100.0 if sum_true > 0 else 0.0
    
    return {
        'model': model,
        'name': name,
        'loss_history': loss_history,
        'MAE': mae,
        'RMSE': rmse,
        'WAPE': wape
    }

results = []
results.append(train_and_eval(LSTMModel(), 'LSTM (Multi-layer)'))
results.append(train_and_eval(BiLSTMModel(), 'Bidirectional LSTM'))
results.append(train_and_eval(GRUModel(), 'GRU (Gated Recurrent)'))

# Plot Loss Convergence Curves
plt.figure(figsize=(10, 5))
for r in results:
    plt.plot(r['loss_history'], label=f"{r['name']} (Final Loss: {r['loss_history'][-1]:.5f})")
plt.title('Training Loss Convergence Comparison (15 Epochs)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Benchmark Results Summary
summary_df = pd.DataFrame([
    {'Architecture': r['name'], 'MAE (units)': round(r['MAE'], 4), 'RMSE (units)': round(r['RMSE'], 4), 'WAPE (%)': round(r['WAPE'], 2)}
    for r in results
])
print('\n=== MODEL BENCHMARK COMPARISON TABLE ===')
print(summary_df.to_string(index=False))


## 6. Checkpoint Download
Download the trained model checkpoint (`demand_lstm.pt`) and fitted `scaler.pkl` to your local machine.


In [ ]:
# Save Checkpoints
best_model = results[0]['model']  # Multi-layer LSTM
torch.save(best_model.state_dict(), 'demand_lstm.pt')
joblib.dump(scaler, 'scaler.pkl')
print('Checkpoints saved: demand_lstm.pt and scaler.pkl')

try:
    from google.colab import files
    print('Downloading weights...')
    files.download('demand_lstm.pt')
    files.download('scaler.pkl')
except Exception as e:
    print('Local environment detected, weights saved in workspace directory.')
